In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool
from langchain_groq import ChatGroq

# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
)

# Create agent with PII Middleware
agent = create_agent(
    model=llm,
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [2]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I’m happy to help! To keep your information safe, I won’t need your full card number—just a brief description of the issue you’re experiencing (e.g., a problem with a recent transaction, a question about your account, a billing dispute, etc.).  

If you’d like me to look up your account, I can do that using the email address you provided, but please confirm that you’re comfortable with me accessing that information. Let me know how I can assist you, and we’ll take it from there.


In [ ]:
# so in the above resposne, the email and credit card number should be redacted/masked in the model's response.
result

# so this emnail has been chanegd to [REDACTED_EMAIL] see below example. 

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='7223ca67-0f30-4894-b770-ebbff665b581'),
  AIMessage(content='I’m happy to help!\u202fTo keep your information safe, I won’t need your full card number—just a brief description of the issue you’re experiencing (e.g., a problem with a recent transaction, a question about your account, a billing dispute, etc.).  \n\nIf you’d like me to look up your account, I can do that using the email address you provided, but please confirm that you’re comfortable with me accessing that information. Let me know how I can assist you, and we’ll take it from there.', additional_kwargs={'reasoning_content': "The user is providing personal data: email and partial card number. They are asking for help. We need to see if they are requesting assistance that requires accessing personal data. Possibly they want account assistance. We must not stor

In [4]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
    
except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content
